In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data_path = '/content/drive/MyDrive/Marathon/data/'

In [18]:
import pandas as pd
import numpy as np
import re

def clean_brand_column(df):
    """Identifie les marques via IDs/modèles et standardise les noms."""
    mask_stellantis = (df['brand'] == '27006279306642')
    df['brand'] = np.where(mask_stellantis & df['modele'].str.contains('^2|^3|^4|^5', regex=True), 'Peugeot',
                  np.where(mask_stellantis & df['modele'].str.contains('^C3', case=False, regex=True), 'Citroen', df['brand']))

    mask_id = (df['brand'] == '26835333024274')
    df['brand'] = np.where(mask_id & df['modele'].str.contains('ypsilon', case=False), 'Lancia',
                  np.where(mask_id & df['modele'].str.contains('audi|q2', case=False), 'Audi', df['brand']))

    mapping_marques = {
        'Audi AGC': 'AUDI', 'LeadGen Mitsubishi': 'Mitsubishi', 'Lexus WCB': 'LEXUS',
        'Nissan Lead Gen': 'Nissan', 'Renault Annalect': 'Renault', 'Vidéo Live Volkswagen': 'Volkswagen',
        'Mercedes-benz': 'Mercedes', 'Mercedes-Benz': 'Mercedes', 'Mercedes-Benz ': 'Mercedes',
        'VW Utilitaires':'VWU', 'Alpine France': 'Alpine', '25091010376082':'Honda',
        '25382330714770':'AUDI', '26881567913874':'Hyundai', '28121657398034':'Skoda',
        '29481365320082':'Volkswagen', '29576441776146':'VWU', '29577564225426':'Seat',
        '29577629036690':'CUPRA', 'Audi':'AUDI'
    }
    df['brand'] = df['brand'].replace(mapping_marques)
    return df[df['brand'] != 'Test E Car']

def clean_model_column(df):
    """
    Nettoie et harmonise les noms de modèles.
    Adaptation stricte du script original.
    """
    # 1. Nettoyage textuel de base (équivalent à clean_model_names)
    def clean_text(name):
        if not isinstance(name, str):
            return name

        # Mise en minuscule, retrait guillemets, strip, replace ë
        name = name.lower().replace('"', '').strip().replace('ë', 'e')

        # Suppression des années
        name = re.sub(r'\b\d{4}\b', '', name)

        # Suppression des mots "bruit"
        noise_words = [
            'hybrid', 'hybride', 'phev', 'ev', 'electric', 'electrique', 'elettrica',
            'facelift', 'edition', 'copa', 'fr', 'style', 'life', 'vw edition',
            'sportback', 'tfsie', 'berline', 'sw', 'vo', 'phase 1', 'l', 'xl', '204 ch'
        ]
        pattern = r'\b(' + '|'.join(noise_words) + r')\b'
        name = re.sub(pattern, '', name)

        # Suppression des marques
        brands = ['volkswagen', 'audi', 'seat', 'cupra', 'skoda', 'hyundai', 'citroen', 'opel']
        brand_pattern = r'\b(' + '|'.join(brands) + r')\b'
        name = re.sub(brand_pattern, '', name)

        # Nettoyage final espaces et tirets
        name = re.sub(r'\s+', ' ', name).strip()
        name = name.replace('- ', '-').strip('-')

        return name.capitalize()

    # Application du nettoyage de base
    df.loc[:, 'modele'] = df['modele'].apply(clean_text)

    # 2. Remplacements spécifiques (Mapping complet restauré)
    corrections = {
        'C3a': 'C3 Aircross',
        'C5a': 'C5 Aircross',
        'Id.3 id.': 'Id.3',
        'Id.4 id.': 'Id.4',
        'Born batterie': 'Born',
        'Born 230 ch batterie': 'Born',
        'Frontera-hev': 'Frontera',
        'Frontera hev': 'Frontera',
        'Nouvelle leon': 'Leon',
        'Nouvelle ateca': 'Ateca',
        'Ateca xperience': 'Ateca',
        'Ev3 100% électrique': 'Ev3',
        'Ev6 100% électrique': 'Ev6',
        'Ypsilon ibrida': 'Ypsilon',
        'Topolino dolcevita': 'Topolino'
    }
    df.loc[:, 'modele'] = df['modele'].replace(corrections)

    # 3. Suppression des valeurs invalides
    valeurs_invalides = ['Gamme', '4x4', 'Leasing social', 'Carrousel', '']
    df = df[~df['modele'].isin(valeurs_invalides)].copy()

    # 4. Nettoyage Regex des termes techniques restants (Etape restaurée)
    mots_techniques = r' (hev|ibrida|100% électrique|max|ch|batterie|230 ch|volkswagen|audi|seat|cupra)'
    df.loc[:, 'modele'] = df['modele'].str.replace(mots_techniques, '', regex=True, case=False)

    # 5. Harmonisation finale (Ec -> C et Suffixes)
    df.loc[:, 'modele'] = df['modele'].str.replace(r'^Ec', 'C', regex=True)

    suffixes = {
        r'-e$': '',
        r' e$': '',
        r' 6.1$': '',
        r' combi$': '',
        r' allspace$': ''
    }
    for pat, rep in suffixes.items():
        df.loc[:, 'modele'] = df['modele'].str.replace(pat, rep, regex=True, case=False)

    # Dernier retrait d'espaces et mise en majuscule
    df.loc[:, 'modele'] = df['modele'].str.strip().str.capitalize()

    return df

def impute_energy(df):
    """
    Remplit les énergies indéterminées pour les modèles ciblés.
    Ne modifie pas les labels 'non_déterminé' pour les autres modèles.
    """
    # Ton dictionnaire métier
    mapping_manuel = {
        'T-roc': 'diesel',
        'Tiguan': 'hybride',
        'Arona': 'essence',
        'Formentor': 'hybride',
        'Ateca': 'diesel',
        'Leon': 'diesel',
        'Q2': 'essence'
    }

    # Liste des variantes rencontrées dans tes données
    labels_inconnus = ['non_déterminé', 'non déterminé']

    for mod, eng in mapping_manuel.items():
        # On utilise .isin() pour attraper les deux variantes (avec ou sans underscore)
        # sans avoir besoin de renommer toute la colonne au préalable.
        mask = (df['energie'].isin(labels_inconnus)) & (df['modele'] == mod)
        df.loc[mask, 'energie'] = eng

    return df



In [19]:
#CHARGER LES DONNEES SELON LE NOM DE VOS VARIABLES

df_conso = pd.read_parquet(data_path+'choix_vehicule.parquet')


#SUPPRIMER LES COLONNES DONT NOUS N'AVONS PAS BESOIN
conso = df_conso.drop(columns=['reprise_energie', 'reprise_annee', "boite_de_vitesse",
                                "modele_concurrence", "reprise_modele"]).copy()

#APPELER LES FONCTIONS DE NETTOYAGE POUR CHAQUE COLONNE A NETTOYER
conso = clean_brand_column(conso)
conso = conso[conso['reprise_couleur'] != "NC"]
conso = clean_model_column(conso)

#NETTOYAGE A LA MAIN POUR LES COLONNES SECONDAIRES
conso = conso[conso['departement'].astype(str).str.isnumeric()]
conso['reprise_marque'] = conso['reprise_marque'].replace({'autre_marque': 'Autre', 'pas de reprise': 'Sans', 'reprise_non': 'Sans'})
conso['marque_concurrence'] = (conso['marque_concurrence']
                               .replace({'concurrence__non': 'Sans', 'concurrence__autre_marque': 'Sans', 'Mercedes benz': 'Mercedes'})
                               .str.strip().str.capitalize())

#IMPUTATION DE LA COLONNE ENERGIE
conso = impute_energy(conso)

#DATA_FINAL
print(conso)
#for col in conso.columns :
  #display(conso[col].value_counts())
#for col in conso.columns :
  #display(conso[col].unique())



        annee  mois  civilite departement       brand       modele  \
index                                                                
2999     2023    11  monsieur          93     Citroen  C3 aircross   
2        2024     8  monsieur          64     Citroen  C3 aircross   
24       2024     8  monsieur          27     Citroen           C3   
74       2024     8    madame          73     Citroen           C3   
128      2024     8  monsieur          27     Citroen          Ami   
...       ...   ...       ...         ...         ...          ...   
122303   2025     6  monsieur          78        Seat        Arona   
122459   2025     6  monsieur          58       CUPRA         Born   
122513   2025     6  monsieur          13  Volkswagen         Polo   
122981   2025     6    madame          13       CUPRA    Formentor   
122982   2025     6    madame          13        AUDI           A3   

           energie  usage presentation_video reprise_marque reprise_couleur  \
index     